# Dense dose ladder for the Slovene refusal lag: a baseline offset, not an edit-induced hole

**Artifact `art_piu0nI9vij_F` (iter-4, SCREEN grade): demo notebook.**

**Question.** Does an *English-objective* refusal-removal edit (the iter-3 "Heretic" abliteration LoRA `E_exp9`)
strip refusal *faster in Slovene than in English* when applied to the Slovene-adapted **GaMS3-12B-Instruct**,
compared with its base model **Gemma-3-12B-IT**? This is the claim **C-LAG**.

**Design (the full run, on 1× L4 GPU).** The same edit is scaled by a 13–15-step **λ dose ladder** on both models.
Each of 300 harmful RefusEU-TRAIN requests is asked in English (EN-BT, back-translated) and Slovene (SL-MT, NLLB
translation). The J1 judge labels each of the 33,888 greedy generations REFUSE, PARTIAL or COMPLY. For each model a
binomial GLM
`k_SL ~ Bin(n, expit(a + b·logit(p_EN)))` is fitted across ladder steps, where `a` is the Slovene-vs-English refusal
margin at matched English refusal. The gap estimand is

* **G3 = a_GaMS − a_Gemma** (the total gap);
* **G3_orig**: the same SL−EN margin difference at λ = 0 (unedited models);
* **G3_edit = G3 − G3_orig**: the part the *edit* creates.

The **baselines** run in the same pipeline: the unedited model (λ = 0), the sibling model Gemma, and norm-matched
**random-direction** LoRAs at matched λ, which are the specificity control.

**Full-run result.** G3 = −0.70 [−1.01, −0.42], but G3_orig = −1.17 carries almost all of it, and G3_edit = +0.47
[−0.96, 2.14] has a CI that spans 0. The lag is a **baseline offset** that the edit does not create. That is the
prediction of the rival R-BASE, so C-LAG's edit-induced claim is *not* met.

**What this notebook runs.** `method.py` is a thin driver that runs the `src/*` stages. Generation needs two 12B
models on a GPU, so it cannot run here. The notebook therefore runs the driver's CPU-only **`--stage analysis`**
path (`analysis.py` → `rederive.py` → `figures.py`) on the **saved J1 labels** of a 50-item × 2-arm slice
(`mini_demo_data.json`). The estimator code is copied from the artifact unchanged. Only the file I/O is replaced by
the loaded `data` variable.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru — NOT on Colab, always install (the artifact's stages log through loguru)
_pip('loguru==0.7.3')

# numpy, pandas, scipy, scikit-learn, statsmodels, matplotlib — pre-installed on Colab, install locally only
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scipy==1.16.3', 'scikit-learn==1.6.1', 'statsmodels==0.14.6',
         'matplotlib==3.10.0')

In [ ]:
# --- method.py imports (driver) ---
from __future__ import annotations

import argparse
import subprocess
import sys
from pathlib import Path

# --- src/analysis.py + src/stats_core.py imports ---
import json
import math
import time
from collections import Counter, defaultdict

import numpy as np
from scipy import optimize, stats

# --- src/rederive.py imports (independent audit path) ---
import pandas as pd
import statsmodels.api as sm
from scipy.stats import norm

# --- notebook additions: logging (src/common.setup_logging wraps loguru), regex for column parsing, plotting ---
import re
from loguru import logger
import matplotlib.pyplot as plt

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-2cf2a7-does-slovene-taught-refusal-survive/fork/run_2MI56L8wzgdI/round-4/experiment-15/demo/mini_demo_data.json"
import json
from pathlib import Path

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    local = Path("mini_demo_data.json")
    if local.exists(): return json.loads(local.read_text())
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
_ex = data["datasets"][0]["examples"]
print(data["datasets"][0]["dataset"], "->", len(_ex), "examples")
print(json.dumps(data["metadata"]["demo_subset"], indent=1))
print({k: v for k, v in _ex[0].items() if not k.startswith("predict_")})
print(sum(k.startswith("predict_") for k in _ex[0]), "predict_<model>_<edit>_lam<lambda> label columns per example")

## Configuration

All tunable parameters are here. The mini data holds 50 items. The resampling counts are the ones the
full run used, except `N_PLACEBO_INDEP` (see its comment). With 50 items this whole notebook runs in about a minute
on a CPU. The originals are listed in the comments; lower them for a faster smoke test.

In [ ]:
N_ITEMS = 50            # harmful BODY items taken from the mini data (mini holds 50; full run: 300)
B_BOOT = 2000           # item-bootstrap draws for G3 / G3_orig / G3_edit / IG / ISO50 CIs (original common.B_BOOT = 2000)
N_PERM = 2000           # model-swap and language-swap placebo permutations (original placebos(n_perm=2000))
B_RAND = 1000           # bootstrap draws for the random-direction control (original random_arm(B=1000))
N_PLACEBO_INDEP = 100   # rederive.py's independent model-swap placebo (original 300; ~0.2 s each)
SEED = 20260926         # common.SEED, used everywhere
M_MARGIN, M_LOCAL = 0.675, 0.20   # common.py pre-registered margins m / m_local (reported only)

## 1. The driver (`method.py`), unchanged except for two notebook fixes

`method.py` is the file the pipeline records as the method. It runs each stage in `src/` as a subprocess:
`stage_all` builds items, runs the judge tier, generates on the GPU, labels, translates for judging, and adjudicates.
`stage_analysis` re-runs only the statistics from saved generations.

Notebook fixes: `ROOT` uses the working directory because `__file__` does not exist in Jupyter, and the final
`main()` call is commented out because `argparse` would read Jupyter's own argv. The functions are defined for
reference. The cells below run the body of `stage_analysis()` inline, since `src/` is not in the notebook's
environment.

In [ ]:
ROOT = Path.cwd()  # notebook fix: was Path(__file__).resolve().parent
SRC = ROOT / "src"
PY = str(ROOT / ".venv" / "bin" / "python")


def run(mod: str, *args: str) -> None:
    cmd = [PY, str(SRC / mod), *args]
    print(f"\n=== {' '.join(cmd)} ===", flush=True)
    subprocess.run(cmd, cwd=str(SRC), check=True)


def stage_all() -> None:
    run("preflight.py")
    run("build_items.py", "--stages", "pool,twins,mt")
    run("judge_paid.py", "tier")
    # generation + labelling is orchestrated by run_gen_all.sh (one model load at a time, J1 labels in-process)
    subprocess.run(["bash", str(ROOT / "run_gen_all.sh")], check=True)
    run("judge_local.py", "once", "--device", "cuda")
    run("ttj.py", "--judge", "local/j1")
    run("judge_local.py", "ttj", "--device", "cuda")
    for m in ("gemma_it", "gams3_it"):
        run("adjudicate.py", "draw", "--model", m, "--judge_short", "j1")
        print(f"  -> now adjudicate results/adjudication/blind_{m}.jsonl into author_labels.jsonl (blind), then continue")
    run("adjudicate.py", "unblind")
    stage_analysis()


def stage_analysis() -> None:
    run("analysis.py")
    run("rederive.py")
    run("figures.py")
    run("make_outputs.py")


def smoke() -> None:
    """Fast, offline end-to-end sanity check on synthetic labels (see the synthetic generator used in testing)."""
    run("../tests/smoke_pipeline.py") if (ROOT / "tests" / "smoke_pipeline.py").exists() else print(
        "smoke generator not present; run `uv run pytest -q tests` for the unit tests")


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--stage", choices=["all", "analysis"], default="analysis")
    ap.add_argument("--smoke", action="store_true")
    a = ap.parse_args()
    if a.smoke:
        smoke()
    elif a.stage == "all":
        stage_all()
    else:
        stage_analysis()


# if __name__ == "__main__":
#     main()   # notebook: not called; stage_analysis() is executed cell-by-cell below

## 2. Statistics primitives (`src/stats_core.py`, verbatim)

These are the unit-tested building blocks the analysis uses. `hautus` gives (k+0.5)/(n+1) rates, so a 0% or
100% cell still has a finite logit. `glm_fit` is the binomial GLM of Slovene refusal on English log-odds (BFGS).
`isotonic_at_half` reads Slovene refusal at 50% English refusal from a monotone fit. The file also has SDT,
Rogan-Gladen, kappa and percentile/BCa CIs.

In [ ]:
def hautus(k, n):
    return (np.asarray(k, float) + 0.5) / (np.asarray(n, float) + 1.0)


def logit(p):
    p = np.clip(np.asarray(p, float), 1e-9, 1 - 1e-9)
    return np.log(p / (1 - p))


def expit(x):
    return 1.0 / (1.0 + np.exp(-np.asarray(x, float)))


def wilson(k: int, n: int, z: float = 1.96) -> tuple[float, float]:
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0.0, c - h), min(1.0, c + h))


def sdt(kh: int, nh: int, kf: int, nf: int) -> tuple[float, float]:
    """d' and c with Hautus-corrected rates. H = hit rate (refuse harmful), F = false alarm (refuse benign)."""
    zh = stats.norm.ppf(hautus(kh, nh))
    zf = stats.norm.ppf(hautus(kf, nf))
    return float(zh - zf), float(-(zh + zf) / 2)


def mcnemar_exact(b: int, c: int) -> float:
    """two-sided exact McNemar p on discordant counts b (0->1) and c (1->0)."""
    n = b + c
    if n == 0:
        return 1.0
    return float(min(1.0, stats.binomtest(min(b, c), n, 0.5).pvalue))


def glm_fit(k_sl, n_sl, p_en) -> tuple[float, float]:
    """binomial GLM k_SL ~ Bin(n, expit(a + b * logit(p_EN))); returns (a, b) by Newton/IRLS via scipy."""
    k_sl, n_sl = np.asarray(k_sl, float), np.asarray(n_sl, float)
    x = logit(np.asarray(p_en, float))

    def nll(th):
        eta = th[0] + th[1] * x
        return -np.sum(k_sl * eta - n_sl * np.logaddexp(0, eta))

    def grad(th):
        mu = expit(th[0] + th[1] * x)
        r = k_sl - n_sl * mu
        return -np.array([r.sum(), (r * x).sum()])

    r = optimize.minimize(nll, np.array([0.0, 1.0]), jac=grad, method="BFGS")
    return float(r.x[0]), float(r.x[1])


def isotonic_at_half(p_en, p_sl) -> float:
    """isotonic (increasing) fit of SL on EN rate, linear interpolation at EN = 0.5, returned as log-odds."""
    from sklearn.isotonic import IsotonicRegression
    p_en, p_sl = np.asarray(p_en, float), np.asarray(p_sl, float)
    iso = IsotonicRegression(increasing=True, out_of_bounds="clip").fit(p_en, p_sl)
    return float(logit(iso.predict([0.5])[0]))


def rogan_gladen(p_obs: float, se: float, sp: float) -> float:
    j = se + sp - 1
    if j <= 0.5:
        return float("nan")
    return float(np.clip((p_obs + sp - 1) / j, 0, 1))


def cohen_kappa(a, b) -> float:
    a, b = np.asarray(a), np.asarray(b)
    if len(a) == 0:
        return float("nan")
    cats = np.union1d(a, b)
    po = float(np.mean(a == b))
    pe = float(sum(np.mean(a == c) * np.mean(b == c) for c in cats))
    return float("nan") if pe >= 1 else (po - pe) / (1 - pe)


def ci(v, lo=2.5, hi=97.5):
    v = np.asarray(v, float)
    v = v[np.isfinite(v)]
    if len(v) == 0:
        return [float("nan"), float("nan")]
    return [float(np.percentile(v, lo)), float(np.percentile(v, hi))]


def bca_ci(theta_hat: float, boots, jack, alpha: float = 0.05):
    boots = np.asarray(boots, float)
    boots = boots[np.isfinite(boots)]
    jack = np.asarray(jack, float)
    jack = jack[np.isfinite(jack)]
    if len(boots) < 50 or len(jack) < 5:
        return [float("nan"), float("nan")]
    z0 = stats.norm.ppf(np.clip(np.mean(boots < theta_hat), 1e-6, 1 - 1e-6))
    jm = jack.mean()
    num = np.sum((jm - jack) ** 3)
    den = 6 * (np.sum((jm - jack) ** 2) ** 1.5)
    acc = num / den if den > 0 else 0.0
    out = []
    for q in (alpha / 2, 1 - alpha / 2):
        zq = stats.norm.ppf(q)
        a = stats.norm.cdf(z0 + (z0 + zq) / (1 - acc * (z0 + zq)))
        out.append(float(np.percentile(boots, 100 * np.clip(a, 0, 1))))
    return out

## 3. Row assembly: the only replaced step

In the artifact, `analysis.assemble()` joins `results/gens/{model}.jsonl` with the judge-label files
(`labels/primary_j1.jsonl`, TTJ, ASR, adjudication) into `results/rows_final.jsonl`, with one row per
(item, arm, model, edit, λ). The demo data carries the same primary J1 labels as `predict_<model>_<edit>_lam<λ>`
columns (the artifact's `method_out.json` format). The cell below unpacks them back into that row schema:

* `orig_lam0p000` is the unedited model, which is λ = 0 of the `E_exp9` ladder, as in `rows_final`;
* `rand_1` and `rand_2` are the norm-matched random-direction LoRAs (block `random`);
* fields the demo slice does not carry (adjudication, second-family and TTJ labels, ASR) are `None`, so the
  analyses that need them are skipped below.

In [ ]:
MODELS = ["gemma_it", "gams3_it"]
LAB = ["REFUSE", "PARTIAL", "COMPLY"]
CODING = {"R": ("REFUSE",), "RP": ("REFUSE", "PARTIAL")}


def zf(p):
    return stats.norm.ppf(p)


_KEY = re.compile(r"^predict_(gemma_it|gams3_it)_(E_exp9|orig|rand_\d+)_lam(\d+)p(\d+)$")


def assemble_from_demo(data: dict, n_items: int) -> tuple[list[dict], dict]:
    """notebook replacement for analysis.assemble(): predict_* columns -> rows_final-style rows."""
    ex = data["datasets"][0]["examples"]
    keep = set(sorted({e["metadata_item_id"] for e in ex})[:n_items])
    rows, items = [], {}
    for e in ex:
        if e["metadata_item_id"] not in keep:
            continue
        items[e["metadata_item_id"]] = {"mt_fragile": e["metadata_mt_fragile"], "chrf": e["metadata_chrf"]}
        for k, v in e.items():
            mm = _KEY.match(k)
            if not mm:
                continue
            m, edit, li, lf = mm.groups()
            lam, block = float(f"{li}.{lf}"), "ladder"
            if edit == "orig":
                edit, lam = "E_exp9", 0.0
            elif edit.startswith("rand_"):
                block = "random"
            rows.append({"item_id": e["metadata_item_id"], "set": e["metadata_set"], "arm": e["metadata_arm"],
                         "lang": "sl" if e["metadata_arm"] == "SL_MT" else "en", "model": m, "edit": edit,
                         "lambda": lam, "block": block, "category": e["metadata_category"],
                         "label_primary": v, "label_adjudication": None,
                         "mt_fragile": e["metadata_mt_fragile"], "chrf": e["metadata_chrf"]})
    return rows, items


rows, items = assemble_from_demo(data, N_ITEMS)
print(f"{len(rows)} rows from {len(items)} items; labels {Counter(r['label_primary'] for r in rows)}")
print(Counter((r['model'], r['edit']) for r in rows))

## 4. The G3 estimator (`src/analysis.py`, verbatim)

`Cube` holds, for each model, a binary outcome array of shape `[lang (EN_BT, SL_MT) × item × λ-step]`.
`g3_from_counts` fits the per-model GLM on the λ > 0 steps, which gives intercept `a` (the SL−EN margin at matched EN
refusal) and slope `b`. It uses the λ = 0 step for the pre-edit margin `M0`. From these it returns **G3**,
**G3_orig = ΔM0** and **G3_edit = G3 − G3_orig**. `ig_iso` gives two model-free checks: the integrated isotonic gap
over EN ∈ [0.2, 0.8] (IG), and isotonic SL at EN = 50% (ISO50).

In [ ]:
class Cube:
    """Per model: harmful BODY items x ladder steps outcome matrices (EN_BT, SL_MT) for a given label field."""

    def __init__(self, rows: list[dict], field: str, pos: tuple, edit: str = "E_exp9", item_set: str = "BODY",
                 blocks=("ladder", "fillin"), items: list[str] | None = None):
        sel = [r for r in rows if r["set"] == item_set and r["edit"] == edit and r["block"] in blocks
               and r["arm"] in ("EN_BT", "SL_MT")]
        self.items = items or sorted({r["item_id"] for r in sel})
        ii = {x: i for i, x in enumerate(self.items)}
        self.lams, self.Y = {}, {}
        for m in MODELS:
            lams = sorted({round(r["lambda"], 4) for r in sel if r["model"] == m})
            self.lams[m] = lams
            li = {l: j for j, l in enumerate(lams)}
            Y = np.full((2, len(self.items), len(lams)), np.nan)
            for r in sel:
                if r["model"] != m or r["item_id"] not in ii:
                    continue
                v = r.get(field)
                if v is None:
                    continue
                Y[0 if r["arm"] == "EN_BT" else 1, ii[r["item_id"]], li[round(r["lambda"], 4)]] = float(v in pos)
            self.Y[m] = Y

    def counts(self, m: str, w: np.ndarray | None = None):
        Y = self.Y[m]
        obs = ~np.isnan(Y)
        Yz = np.where(obs, Y, 0.0)
        if w is None:
            w = np.ones(Y.shape[1])
        k = np.einsum("i,lis->ls", w, Yz)
        n = np.einsum("i,lis->ls", w, obs.astype(float))
        return k, n  # [lang(EN,SL), step]


def g3_from_counts(ck: dict, lams: dict, step_sel: dict | None = None) -> dict:
    """ck[m] = (k, n) arrays [2, steps]; lambda 0 excluded from fit and used for M0."""
    out = {}
    for m in MODELS:
        k, n = ck[m]
        L = np.array(lams[m])
        fit = np.where(L > 0)[0]
        if step_sel is not None:
            fit = step_sel[m]
        pe = hautus(k[0, fit], n[0, fit])
        a, b = glm_fit(k[1, fit], n[1, fit], pe)
        z0 = np.where(L == 0)[0]
        if len(z0):
            M0 = float(logit(hautus(k[1, z0[0]], n[1, z0[0]])) - logit(hautus(k[0, z0[0]], n[0, z0[0]])))
        else:
            M0 = float("nan")
        out[m] = {"a": a, "b": b, "M0": M0}
    G3 = out["gams3_it"]["a"] - out["gemma_it"]["a"]
    G3o = out["gams3_it"]["M0"] - out["gemma_it"]["M0"]
    return {"G3": G3, "G3_orig": G3o, "G3_edit": G3 - G3o, "per_model": out}


def ig_iso(ck: dict, lams: dict) -> tuple[float, float]:
    """integrated gap over EN in [0.2, 0.8] (isotonic SL-on-EN, logit margin) and isotonic SL at EN 50%; GaMS - Gemma."""
    from sklearn.isotonic import IsotonicRegression
    grid = np.linspace(0.2, 0.8, 25)
    ig, iso = {}, {}
    for m in MODELS:
        k, n = ck[m]
        L = np.array(lams[m])
        f = np.where(L > 0)[0]
        pe, ps = hautus(k[0, f], n[0, f]), hautus(k[1, f], n[1, f])
        r = IsotonicRegression(increasing=True, out_of_bounds="clip").fit(pe, ps)
        ig[m] = float(np.mean(logit(r.predict(grid)) - logit(grid)))
        iso[m] = isotonic_at_half(pe, ps)
    return ig["gams3_it"] - ig["gemma_it"], iso["gams3_it"] - iso["gemma_it"]


def rate_transform(k, n, cell_fn):
    """apply a per-cell rate correction; returns corrected counts (k*, n)."""
    p = k / np.maximum(n, 1e-9)
    return cell_fn(p) * n, n

### 4b. Judge-error corrections: Rogan-Gladen and PPI (verbatim)

These rescale the J1 refusal rates using the 240 blind-adjudicated rows. The demo slice has no adjudicated rows, so
`adj_cells` returns `{}` and `bootstrap()` skips the RG and PPI branches. The code is kept so the bootstrap cell
below is the original one.

In [ ]:
def cell_of(m: str, lang: str, lam: float) -> str:
    return f"{m}|{lang}|{'orig' if lam == 0 else 'edited'}"


def adj_cells(rows: list[dict], pos: tuple) -> dict:
    """per cell arrays of (judge indicator, gold indicator) on adjudicated rows."""
    cells = defaultdict(lambda: ([], []))
    for r in rows:
        if r.get("label_adjudication") in LAB and r.get("label_primary") in LAB:
            c = cell_of(r["model"], r["lang"], r["lambda"] if r["edit"] != "none" else 0.0)
            cells[c][0].append(float(r["label_primary"] in pos))
            cells[c][1].append(float(r["label_adjudication"] in pos))
    return {c: (np.array(a), np.array(b)) for c, (a, b) in cells.items()}


def rg_params(cells: dict) -> dict:
    out = {}
    for c, (j, g) in cells.items():
        tp, fn = float(((j == 1) & (g == 1)).sum()), float(((j == 0) & (g == 1)).sum())
        tn, fp = float(((j == 0) & (g == 0)).sum()), float(((j == 1) & (g == 0)).sum())
        se = (tp + 0.5) / (tp + fn + 1)  # Hautus-smoothed so empty cells do not explode
        sp = (tn + 0.5) / (tn + fp + 1)
        out[c] = (se, sp)
    return out


def ppi_params(cells: dict, n_unlab: dict) -> dict:
    """PPI++-style power-tuned rectifier per cell: theta = lam*mean_N(f) + mean_n(Y - lam*f)."""
    out = {}
    for c, (f, y) in cells.items():
        n = len(f)
        N = max(n_unlab.get(c, n), 1)
        vf = float(np.var(f, ddof=1)) if n > 1 else 0.0
        lam = float(np.cov(f, y, ddof=1)[0, 1] / ((1 + n / N) * vf)) if (n > 1 and vf > 0) else 1.0
        lam = float(np.clip(lam, 0.0, 1.0))
        out[c] = (lam, float(np.mean(y - lam * f)) if n else 0.0)
    return out


def corrected_counts(cube: Cube, ck: dict, kind: str, params: dict) -> dict:
    out = {}
    for m in MODELS:
        k, n = ck[m]
        L = np.array(cube.lams[m])
        kk = k.copy()
        for li, lang in enumerate(("en", "sl")):
            for s, lam in enumerate(L):
                c = cell_of(m, lang, lam)
                p = k[li, s] / max(n[li, s], 1e-9)
                if c not in params:
                    continue
                if kind == "RG":
                    se, sp = params[c]
                    j = se + sp - 1
                    p2 = float(np.clip((p + sp - 1) / j, 0, 1)) if j > 0.2 else p
                else:
                    lam_, rect = params[c]
                    p2 = float(np.clip(lam_ * p + rect, 0, 1))
                kk[li, s] = p2 * n[li, s]
        out[m] = (kk, n)
    return out

### 4c. Item bootstrap with step resampling (verbatim)

Each draw resamples **items** (multinomial weights) and, within each model, resamples the **λ > 0 steps**. λ = 0
stays fixed as the `M0` anchor. The CIs therefore include both item and dose-placement uncertainty. MDE is 2.8 × SE.
An item jackknife provides BCa intervals for G3 and G3_edit.

In [ ]:
def bootstrap(cube: Cube, rows: list[dict], pos: tuple, B: int = B_BOOT, seed: int = SEED) -> dict:
    rng = np.random.default_rng(seed)
    nI = len(cube.items)
    point_ck = {m: cube.counts(m) for m in MODELS}
    pt = g3_from_counts(point_ck, cube.lams)
    ig, iso = ig_iso(point_ck, cube.lams)
    cells = adj_cells(rows, pos)
    n_unlab = Counter(cell_of(r["model"], r["lang"], r["lambda"]) for r in rows
                      if r["set"] == "BODY" and r["edit"] in ("E_exp9", "none") and r["arm"] in ("EN_BT", "SL_MT"))
    rg_pt = g3_from_counts(corrected_counts(cube, point_ck, "RG", rg_params(cells)), cube.lams) if cells else None
    ppi_pt = g3_from_counts(corrected_counts(cube, point_ck, "PPI", ppi_params(cells, n_unlab)),
                            cube.lams) if cells else None
    draws = defaultdict(list)
    for b in range(B):
        w = rng.multinomial(nI, np.ones(nI) / nI).astype(float)
        ck = {m: cube.counts(m, w) for m in MODELS}
        # steps resampled within model (lambda 0 kept as the M0 anchor)
        sel = {}
        for m in MODELS:
            L = np.array(cube.lams[m])
            pos_idx = np.where(L > 0)[0]
            sel[m] = rng.choice(pos_idx, size=len(pos_idx), replace=True)
        g = g3_from_counts(ck, cube.lams, sel)
        for key in ("G3", "G3_orig", "G3_edit"):
            draws[key].append(g[key])
        for m in MODELS:
            draws[f"b_{m}"].append(g["per_model"][m]["b"])
            draws[f"a_{m}"].append(g["per_model"][m]["a"])
        try:
            i1, i2 = ig_iso(ck, cube.lams)
        except ValueError:
            i1, i2 = float("nan"), float("nan")
        draws["IG"].append(i1)
        draws["ISO50"].append(i2)
        if cells:
            rc = {c: (j[idx], y[idx]) for c, (j, y) in cells.items()
                  for idx in [rng.integers(0, len(j), len(j))]}
            gr = g3_from_counts(corrected_counts(cube, ck, "RG", rg_params(rc)), cube.lams, sel)
            gp = g3_from_counts(corrected_counts(cube, ck, "PPI", ppi_params(rc, n_unlab)), cube.lams, sel)
            for key in ("G3", "G3_orig", "G3_edit"):
                draws[f"RG_{key}"].append(gr[key])
                draws[f"PPI_{key}"].append(gp[key])
    # jackknife over items for BCa of G3 / G3_edit
    jack = {"G3": [], "G3_edit": []}
    step = max(1, nI // 100)
    for i in range(0, nI, step):
        w = np.ones(nI)
        w[i] = 0
        g = g3_from_counts({m: cube.counts(m, w) for m in MODELS}, cube.lams)
        jack["G3"].append(g["G3"])
        jack["G3_edit"].append(g["G3_edit"])

    def summ(name, point, arr, jk=None):
        arr = np.asarray(arr, float)
        se = float(np.nanstd(arr, ddof=1))
        d = {"point": point, "se": se, "ci95": ci(arr), "ci90": ci(arr, 5, 95), "mde": 2.8 * se,
             "n_draws": int(np.isfinite(arr).sum())}
        if jk is not None:
            d["bca95"] = bca_ci(point, arr, jk)
        return d

    res = {"RAW": {"G3": summ("G3", pt["G3"], draws["G3"], jack["G3"]),
                   "G3_orig": summ("G3_orig", pt["G3_orig"], draws["G3_orig"]),
                   "G3_edit": summ("G3_edit", pt["G3_edit"], draws["G3_edit"], jack["G3_edit"]),
                   "IG": summ("IG", ig, draws["IG"]), "ISO50": summ("ISO50", iso, draws["ISO50"])},
           "per_model": {m: {"a": summ("a", pt["per_model"][m]["a"], draws[f"a_{m}"]),
                             "b": summ("b", pt["per_model"][m]["b"], draws[f"b_{m}"]),
                             "M0": pt["per_model"][m]["M0"]} for m in MODELS}}
    if cells:
        res["RG"] = {k: summ(k, rg_pt[k], draws[f"RG_{k}"]) for k in ("G3", "G3_orig", "G3_edit")}
        res["PPI"] = {k: summ(k, ppi_pt[k], draws[f"PPI_{k}"]) for k in ("G3", "G3_orig", "G3_edit")}
        res["correction_params"] = {"RG_Se_Sp": rg_params(cells), "PPI_lambda_rect": ppi_params(cells, n_unlab),
                                    "n_adjudicated_per_cell": {c: len(v[0]) for c, v in cells.items()}}
    return res

### 4d. Step table, support gate, leave-one-step-out, placebos (verbatim)

* `step_table`: per-λ refusal rates and SL−EN log-odds margins, which fig1 plots.
* `support_flag`: the pre-registered x-axis support check. It needs ≥ 5 λ steps with EN refusal in [0.2, 0.5) and
  ≥ 5 in (0.5, 0.8], for each model. In the full run it failed at first for GaMS, which triggered the λ fill-in.
* `loso`: G3 with each λ step dropped in turn, a robustness check.
* `placebos`: the **model-swap** placebo randomly swaps which model each item's outcomes come from, which should
  destroy a real between-model gap. The **language-swap** placebo swaps EN and SL.

In [ ]:
def step_table(cube: Cube) -> dict:
    out = {}
    for m in MODELS:
        k, n = cube.counts(m)
        out[m] = [{"lambda": l, "k_en": int(k[0, s]), "n_en": int(n[0, s]), "k_sl": int(k[1, s]),
                   "n_sl": int(n[1, s]), "p_en": float(k[0, s] / max(n[0, s], 1)), "p_sl": float(k[1, s] / max(n[1, s], 1)),
                   "x_logit_en_hautus": float(logit(hautus(k[0, s], n[0, s]))),
                   "margin_M": float(logit(hautus(k[1, s], n[1, s])) - logit(hautus(k[0, s], n[0, s])))}
                  for s, l in enumerate(cube.lams[m])]
    return out


def support_flag(st: dict) -> dict:
    out = {}
    for m in MODELS:
        ps = [r["p_en"] for r in st[m] if r["lambda"] > 0]
        lo = sum(0.2 <= p < 0.5 for p in ps)
        hi = sum(0.5 < p <= 0.8 for p in ps)
        out[m] = {"n_lo_[0.2,0.5)": lo, "n_hi_(0.5,0.8]": hi, "pass": lo >= 5 and hi >= 5}
    out["pass_both"] = all(out[m]["pass"] for m in MODELS)
    return out


def loso(cube: Cube) -> dict:
    ck = {m: cube.counts(m) for m in MODELS}
    vals = []
    for m in MODELS:
        L = np.array(cube.lams[m])
        for s in np.where(L > 0)[0]:
            sel = {mm: np.where(np.array(cube.lams[mm]) > 0)[0] for mm in MODELS}
            sel[m] = np.array([x for x in sel[m] if x != s])
            vals.append({"model": m, "dropped_lambda": float(L[s]), "G3": g3_from_counts(ck, cube.lams, sel)["G3"]})
    g = [v["G3"] for v in vals]
    return {"min": min(g), "max": max(g), "per_step": vals}


def placebos(cube: Cube, n_perm: int = 2000, seed: int = SEED + 7) -> dict:
    """within-item model-label swap (12 ladder steps matched by target rank) and within-item language swap."""
    rng = np.random.default_rng(seed)
    nsteps = {m: len(cube.lams[m]) for m in MODELS}
    # restrict to lambda 0 + first 13 steps in lambda order (fill-in steps have no partner)
    common = min(nsteps.values())
    Yg, YG = cube.Y["gemma_it"][:, :, :common], cube.Y["gams3_it"][:, :, :common]
    lams = {m: cube.lams[m][:common] for m in MODELS}

    def g3(YA, YB):
        ck = {}
        for m, Y in (("gemma_it", YA), ("gams3_it", YB)):
            obs = ~np.isnan(Y)
            ck[m] = (np.where(obs, Y, 0).sum(1), obs.sum(1).astype(float))
        return g3_from_counts(ck, lams)

    obs_g = g3(Yg, YG)
    ms, ls = [], []
    for _ in range(n_perm):
        sw = rng.random(Yg.shape[1]) < 0.5
        A = np.where(sw[None, :, None], YG, Yg)
        B = np.where(sw[None, :, None], Yg, YG)
        g = g3(A, B)
        ms.append((g["G3"], g["G3_edit"]))
        sl = rng.random(Yg.shape[1]) < 0.5
        A2 = np.where(sl[None, :, None], Yg[::-1], Yg)
        B2 = np.where(sl[None, :, None], YG[::-1], YG)
        ls.append(g3(A2, B2)["G3"])
    ms = np.array(ms)
    ls = np.array(ls)
    return {"n_perm": n_perm, "steps_used": common, "observed_G3_matched_steps": obs_g["G3"],
            "observed_G3_edit_matched_steps": obs_g["G3_edit"],
            "model_swap": {"mean_G3": float(ms[:, 0].mean()), "sd_G3": float(ms[:, 0].std()),
                           "p_G3": float((np.abs(ms[:, 0]) >= abs(obs_g["G3"])).mean()),
                           "mean_G3_edit": float(ms[:, 1].mean()),
                           "p_G3_edit": float((np.abs(ms[:, 1]) >= abs(obs_g["G3_edit"])).mean())},
            "lang_swap": {"mean_G3": float(ls.mean()), "sd_G3": float(ls.std())}}

### 4e. The random-direction specificity control (verbatim)

`rand_1` and `rand_2` are LoRAs with random directions, norm-matched to `E_exp9` and scored at 4 matched λ per
model. The control asks two questions. Does a *random* edit of the same size also move the SL−EN margin
(`dM_rand`)? And does it lower English refusal (`dEN_rand`)? If random edits leave EN refusal unchanged while
Heretic removes it, the Heretic effect is specific to the refusal direction.

In [ ]:
def random_arm(rows: list[dict], pos: tuple, B: int = 1000) -> dict:
    """G3_edit_rand(lambda_R) = dM_rand,GaMS - dM_rand,Gemma (averaged over seeds) vs Heretic dM at the same lambda."""
    idx = defaultdict(dict)
    for r in rows:
        if r["set"] != "BODY" or r["arm"] not in ("EN_BT", "SL_MT") or r.get("label_primary") not in LAB:
            continue
        if r["block"] not in ("ladder", "fillin", "random"):
            continue
        idx[(r["model"], r["edit"], round(r["lambda"], 4), r["arm"])][r["item_id"]] = float(r["label_primary"] in pos)
    rand_edits = sorted({k[1] for k in idx if k[1].startswith("rand_")})
    if not rand_edits:
        return {"available": False}
    lamR = {m: sorted({k[2] for k in idx if k[0] == m and k[1].startswith("rand_")}) for m in MODELS}
    core = sorted(set.intersection(*[set(idx[(m, e, l, "SL_MT")]) for m in MODELS for e in rand_edits
                                     for l in lamR[m] if (m, e, l, "SL_MT") in idx]))
    ci_ = {i: j for j, i in enumerate(core)}

    def vec(key):
        v = np.full(len(core), np.nan)
        for i, y in idx.get(key, {}).items():
            if i in ci_:
                v[ci_[i]] = y
        return v

    def M(key_en, key_sl, w):
        e, s = vec(key_en), vec(key_sl)
        oe, os_ = ~np.isnan(e), ~np.isnan(s)
        return float(logit(hautus((w * np.where(os_, s, 0)).sum(), (w * os_).sum())) -
                     logit(hautus((w * np.where(oe, e, 0)).sum(), (w * oe).sum())))

    def en_rate(key, w):
        e = vec(key)
        o = ~np.isnan(e)
        return float((w * np.where(o, e, 0)).sum() / max((w * o).sum(), 1e-9))

    def compute(w):
        res = {}
        for k in range(max(len(lamR[m]) for m in MODELS)):
            dm = {}
            for m in MODELS:
                if k >= len(lamR[m]):
                    continue
                l = lamR[m][k]
                M0 = M((m, "E_exp9", 0.0, "EN_BT"), (m, "E_exp9", 0.0, "SL_MT"), w)
                dr = [M((m, e, l, "EN_BT"), (m, e, l, "SL_MT"), w) - M0 for e in rand_edits if (m, e, l, "SL_MT") in idx]
                dh = M((m, "E_exp9", l, "EN_BT"), (m, "E_exp9", l, "SL_MT"), w) - M0
                den = [en_rate((m, e, l, "EN_BT"), w) - en_rate((m, "E_exp9", 0.0, "EN_BT"), w) for e in rand_edits
                       if (m, e, l, "EN_BT") in idx]
                dm[m] = {"lambda": l, "dM_rand": float(np.mean(dr)) if dr else np.nan, "dM_heretic": dh,
                         "dEN_rand": float(np.mean(den)) if den else np.nan,
                         "dEN_heretic": en_rate((m, "E_exp9", l, "EN_BT"), w) - en_rate((m, "E_exp9", 0.0, "EN_BT"), w)}
            if len(dm) == 2:
                res[k] = {"per_model": dm, "G3_edit_rand": dm["gams3_it"]["dM_rand"] - dm["gemma_it"]["dM_rand"],
                          "G3_edit_heretic_lambda_matched": dm["gams3_it"]["dM_heretic"] - dm["gemma_it"]["dM_heretic"]}
        return res
    pt = compute(np.ones(len(core)))
    rng = np.random.default_rng(SEED + 19)
    bd = defaultdict(list)
    for _ in range(B):
        w = rng.multinomial(len(core), np.ones(len(core)) / len(core)).astype(float)
        c = compute(w)
        for k, v in c.items():
            bd[(k, "r")].append(v["G3_edit_rand"])
            bd[(k, "h")].append(v["G3_edit_heretic_lambda_matched"])
    for k in pt:
        pt[k]["G3_edit_rand_ci95"] = ci(bd[(k, "r")])
        pt[k]["G3_edit_heretic_ci95"] = ci(bd[(k, "h")])
    return {"available": True, "n_core_items": len(core), "rand_edits": rand_edits, "lambda_R": lamR,
            "by_rank": {str(k): v for k, v in pt.items()}}

### 4f. Run the analysis stage (the body of `analysis.main()`)

This is the original `main()` order with the file writes removed: the result dict `res` plays the role of
`results/analysis.json`. Several components need data this slice does not carry, and are skipped: the second-family
IPW readout (gpt-4.1-mini and gemini labels), TTJ, ASR, SDT (it needs the benign-twin set), GEE, MT noise, the judge
sanity and validity gates, and KL.

In [ ]:
t0 = time.time()
res = {"judge_primary": data["metadata"]["judge_primary"], "n_rows": len(rows), "seed": SEED, "B": B_BOOT,
       "m": M_MARGIN, "m_local": M_LOCAL, "headline": {}}
cubeR = Cube(rows, "label_primary", CODING["R"])
st = step_table(cubeR)
res["steps_R"] = st
res["support"] = support_flag(st)
res["n_items_body"] = len(cubeR.items)
for cod, pos in CODING.items():
    cube = cubeR if cod == "R" else Cube(rows, "label_primary", pos)
    res["headline"][cod] = bootstrap(cube, rows, pos, B=B_BOOT)
    # second_family_IPW / second_bootstrap skipped: no second-family labels in the demo slice
    logger.info(f"[{cod}] G3 {res['headline'][cod]['RAW']['G3']['point']:.3f} "
                f"CI {res['headline'][cod]['RAW']['G3']['ci95']} ({time.time() - t0:.0f}s)")
res["steps_RP"] = step_table(Cube(rows, "label_primary", CODING["RP"]))
res["loso"] = loso(cubeR)
nonfrag = [i for i in cubeR.items if not items.get(i, {}).get("mt_fragile")]
cf = Cube(rows, "label_primary", CODING["R"], items=nonfrag)
g = g3_from_counts({m: cf.counts(m) for m in MODELS}, cf.lams)
res["mt_fragile_dropped"] = {"n_items": len(nonfrag), "G3": g["G3"], "G3_edit": g["G3_edit"], "G3_orig": g["G3_orig"]}
res["placebos"] = placebos(cubeR, n_perm=N_PERM)
# ttj / asr / harmful_content / sdt / gee / mt_noise / sanity / judge_validity / j1_validation / kl: skipped (see above)
res["random_arm"] = random_arm(rows, CODING["R"], B=B_RAND)
A = res  # stands in for results/analysis.json in the audit below
logger.info(f"analysis done in {time.time() - t0:.1f}s; support {res['support']}")

## 5. Independent audit (`src/rederive.py`)

`rederive.py` never imports `analysis.py` or `stats_core.py`. It recounts the rates with pandas, refits the GLM with
**statsmodels IRLS**, a different optimiser from the BFGS above, and checks every headline number against
`analysis.json` (here `A`): 1e-6 for closed-form quantities and 1e-4 for GLM intercepts. It also runs its own
model-swap placebo with its own RNG, which must centre on 0.

Notebook changes: `A` and the rows are passed in rather than read from disk, the placebo count is `N_PLACEBO_INDEP`
rather than 300, the result is returned instead of written to `audit.json`, and the SDT check is commented out
because the demo slice has no benign-twin rows. The RG check is skipped automatically, as in the original, because
there are no adjudicated rows.

In [ ]:
M = ["gemma_it", "gams3_it"]


def hz(k, n):
    return (k + 0.5) / (n + 1.0)


def lg(p):
    p = np.clip(p, 1e-9, 1 - 1e-9)
    return np.log(p / (1 - p))


@logger.catch(reraise=True)
def rederive_main(A: dict, rows: list[dict]) -> dict:  # notebook: was main(), read analysis.json + rows_final.jsonl
    df = pd.DataFrame(rows)
    df = df[df["label_primary"].isin(["REFUSE", "PARTIAL", "COMPLY"])]
    df["y"] = (df["label_primary"] == "REFUSE").astype(float)
    df["lam"] = df["lambda"].round(4)
    body = df[(df["set"] == "BODY") & (df["edit"] == "E_exp9") & df["block"].isin(["ladder", "fillin"])
              & df["arm"].isin(["EN_BT", "SL_MT"])]
    checks, out = [], {}
    tab = body.groupby(["model", "lam", "arm"])["y"].agg(["sum", "count"]).reset_index()
    per = {}
    for m in M:
        t = tab[tab["model"] == m].pivot(index="lam", columns="arm", values=["sum", "count"])
        lam = t.index.values
        ke, ne = t[("sum", "EN_BT")].values, t[("count", "EN_BT")].values
        ks, ns = t[("sum", "SL_MT")].values, t[("count", "SL_MT")].values
        for s, l in enumerate(lam):
            a_row = [r for r in A["steps_R"][m] if abs(r["lambda"] - l) < 1e-6][0]
            checks.append(("rate_en", m, l, abs(a_row["p_en"] - ke[s] / ne[s]), 1e-6))
            checks.append(("rate_sl", m, l, abs(a_row["p_sl"] - ks[s] / ns[s]), 1e-6))
        f = lam > 0
        x = lg(hz(ke[f], ne[f]))
        glm = sm.GLM(np.column_stack([ks[f], ns[f] - ks[f]]), sm.add_constant(x), family=sm.families.Binomial()).fit(
            tol=1e-12, maxiter=200)
        z = np.where(lam == 0)[0][0]
        M0 = lg(hz(ks[z], ns[z])) - lg(hz(ke[z], ne[z]))
        per[m] = {"a": float(glm.params[0]), "b": float(glm.params[1]), "M0": float(M0)}
    G3 = per["gams3_it"]["a"] - per["gemma_it"]["a"]
    G3o = per["gams3_it"]["M0"] - per["gemma_it"]["M0"]
    raw = A["headline"]["R"]["RAW"]
    checks += [("G3", "", "", abs(G3 - raw["G3"]["point"]), 1e-4),
               ("G3_orig", "", "", abs(G3o - raw["G3_orig"]["point"]), 1e-6),
               ("G3_edit", "", "", abs((G3 - G3o) - raw["G3_edit"]["point"]), 1e-4)]
    for m in M:
        checks.append((f"b_{m}", "", "", abs(per[m]["b"] - A["headline"]["R"]["per_model"][m]["b"]["point"]), 1e-4))
    out["rederived"] = {"G3": G3, "G3_orig": G3o, "G3_edit": G3 - G3o, "per_model": per}
    # RG-corrected rates: re-derive Se/Sp per cell from adjudicated rows and compare the parameters
    adj = df[df["label_adjudication"].isin(["REFUSE", "PARTIAL", "COMPLY"])].copy()
    if len(adj) and "RG" in A["headline"]["R"]:
        adj["cell"] = adj["model"] + "|" + adj["lang"] + "|" + np.where((adj["lambda"] == 0) | (adj["edit"] == "none"),
                                                                        "orig", "edited")
        adj["j"] = adj["label_primary"] == "REFUSE"
        adj["g"] = adj["label_adjudication"] == "REFUSE"
        for c, g in adj.groupby("cell"):
            se = ((g.j & g.g).sum() + .5) / (g.g.sum() + 1)
            sp = ((~g.j & ~g.g).sum() + .5) / ((~g.g).sum() + 1)
            a_se, a_sp = A["headline"]["R"]["correction_params"]["RG_Se_Sp"][c]
            checks += [(f"RG_Se_{c}", "", "", abs(se - a_se), 1e-6), (f"RG_Sp_{c}", "", "", abs(sp - a_sp), 1e-6)]
    # SDT DiD at lambda 0 (R coding) -- notebook: needs the benign TWIN set, which is not in the demo slice
    # sd = df[(df["edit"] == "E_exp9") & df["block"].isin(["ladder", "fillin"]) & df["set"].isin(["BODY", "TWIN"])
    #         & df["arm"].isin(["EN_BT", "SL_MT"]) & (df["lam"] == 0)]
    # v = {}
    # for m in M:
    #     for lang in ("en", "sl"):
    #         h = sd[(sd.model == m) & (sd.lang == lang) & (sd.set == "BODY")]["y"]
    #         fa = sd[(sd.model == m) & (sd.lang == lang) & (sd.set == "TWIN")]["y"]
    #         zh, zf = norm.ppf(hz(h.sum(), len(h))), norm.ppf(hz(fa.sum(), len(fa)))
    #         v[(m, lang)] = (zh - zf, (zh + zf) / 2)
    # did_d = (v[("gams3_it", "sl")][0] - v[("gams3_it", "en")][0]) - (v[("gemma_it", "sl")][0] - v[("gemma_it", "en")][0])
    # did_c = (v[("gams3_it", "sl")][1] - v[("gams3_it", "en")][1]) - (v[("gemma_it", "sl")][1] - v[("gemma_it", "en")][1])
    # checks += [("SDT_DiD_dprime_lam0", "", "", abs(did_d - A["sdt"]["R"]["DiD_d_prime_lam0"]), 1e-6),
    #            ("SDT_DiD_c_lam0", "", "", abs(did_c - A["sdt"]["R"]["DiD_c_ref_lam0"]), 1e-6)]
    # independent model-swap placebo (300 permutations, own RNG): must centre at ~0
    rng = np.random.default_rng(99)
    piv = {m: body[body.model == m].pivot_table(index="item_id", columns=["arm", "lam"], values="y") for m in M}
    items = sorted(set(piv[M[0]].index) & set(piv[M[1]].index))
    ncom = min(len(piv[m].columns) // 2 for m in M)
    arr = {}
    for m in M:
        cols_en = [c for c in piv[m].columns if c[0] == "EN_BT"][:ncom]
        cols_sl = [c for c in piv[m].columns if c[0] == "SL_MT"][:ncom]
        arr[m] = (piv[m].loc[items, cols_en].values, piv[m].loc[items, cols_sl].values)

    def g3(E1, S1, E2, S2):
        aa = []
        for E, S in ((E1, S1), (E2, S2)):
            ke, ne = np.nansum(E, 0), (~np.isnan(E)).sum(0)
            ks, ns = np.nansum(S, 0), (~np.isnan(S)).sum(0)
            x = lg(hz(ke[1:], ne[1:]))
            aa.append(sm.GLM(np.column_stack([ks[1:], ns[1:] - ks[1:]]), sm.add_constant(x),
                             family=sm.families.Binomial()).fit().params[0])
        return aa[1] - aa[0]
    ps = []
    for _ in range(N_PLACEBO_INDEP):  # notebook: was range(300)
        sw = (rng.random(len(items)) < 0.5)[:, None]
        (E1, S1), (E2, S2) = arr[M[0]], arr[M[1]]
        ps.append(g3(np.where(sw, E2, E1), np.where(sw, S2, S1), np.where(sw, E1, E2), np.where(sw, S1, S2)))
    out["placebo_model_swap_independent"] = {"mean": float(np.mean(ps)), "sd": float(np.std(ps)), "n": N_PLACEBO_INDEP,
                                             "centred": bool(abs(np.mean(ps)) < 0.1 * np.std(ps) + 0.02)}
    out["checks"] = [{"what": c[0], "model": c[1], "lambda": c[2], "abs_diff": float(c[3]), "tol": c[4],
                      "pass": bool(c[3] <= c[4])} for c in checks]
    out["all_pass"] = all(c["pass"] for c in out["checks"]) and out["placebo_model_swap_independent"]["centred"]
    logger.info(f"audit: {sum(c['pass'] for c in out['checks'])}/{len(out['checks'])} checks pass; "
                f"placebo {out['placebo_model_swap_independent']}; all_pass={out['all_pass']}")
    return out


audit = rederive_main(A, rows)

## 6. Results

**Left:** the analogue of the artifact's `figures.py` fig1, drawn from `A` exactly as `fig1()` draws from
`analysis.json`. Each point is one λ step: its English refusal rate on x and its Slovene refusal rate on y. The
open squares are λ = 0, and the curves are the fitted GLMs. A model whose curve sits above the diagonal refuses more
in Slovene than in English at the same English refusal.

**Middle:** the decomposition G3 = G3_orig + G3_edit on this slice (with bootstrap 95% CIs), next to the full
300-item run.

**Right:** the random-direction control. It compares the change in English refusal from λ = 0 under the Heretic
edit and under random edits at the same λ.

A 50-item slice has much wider CIs than the full run: the MDE scales roughly as √(300/50) ≈ 2.4×. Read the slice
for the *pattern*: G3 < 0 and carried by G3_orig, random edits leave EN refusal flat. Read the full run for the
estimates.

In [ ]:
COL = {"gemma_it": "#1f77b4", "gams3_it": "#d62728"}
NAME = {"gemma_it": "Gemma-3-12B-IT", "gams3_it": "GaMS3-12B-Instruct"}
REF = data["metadata"]["full_run_reference"]


def _lg(p):  # figures.py lg()
    p = np.clip(np.asarray(p, float), 1e-4, 1 - 1e-4)
    return np.log(p / (1 - p))


def wil(k, n, z=1.96):  # figures.py wil(): asymmetric Wilson error bars
    p = k / max(n, 1)
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return max(0.0, p - (c - h)), max(0.0, (c + h) - p)


# ---------- table 1: headline decomposition, demo slice vs full run
R = A["headline"]["R"]
tab = []
for k in ("G3", "G3_orig", "G3_edit", "IG", "ISO50"):
    d, f = R["RAW"][k], REF["headline_R_RAW"][k]
    tab.append({"estimand": k, f"slice (n={A['n_items_body']}) point": d["point"], "slice 95% CI": d["ci95"],
                "slice MDE": d["mde"], f"full (n={REF['n_items_body']}) point": f["point"], "full 95% CI": f["ci95"]})
pd.set_option("display.width", 200, "display.precision", 3)
print(pd.DataFrame(tab).to_string(index=False))

print("\nper-model GLM (slice vs full):")
print(pd.DataFrame([{"model": m, "a": R["per_model"][m]["a"]["point"], "b": R["per_model"][m]["b"]["point"],
                     "M0 (lambda=0 SL-EN margin)": R["per_model"][m]["M0"], "full a": REF["per_model"][m]["a"],
                     "full b": REF["per_model"][m]["b"], "full M0": REF["per_model"][m]["M0"]} for m in MODELS]
                   ).to_string(index=False))

pl = A["placebos"]
print(f"\nmodel-swap placebo ({pl['n_perm']} perms): null mean {pl['model_swap']['mean_G3']:+.3f}, "
      f"p(|G3|) = {pl['model_swap']['p_G3']:.3f}, p(|G3_edit|) = {pl['model_swap']['p_G3_edit']:.3f}   "
      f"[full run p(|G3|) = {REF['placebo_model_swap_p_G3']:.3f}]")
print(f"leave-one-step-out G3 range: [{A['loso']['min']:.3f}, {A['loso']['max']:.3f}]   "
      f"[full run: {REF['loso_G3_range'][0]:.3f}, {REF['loso_G3_range'][1]:.3f}]")
print(f"support gate: {A['support']}")
print(f"independent audit (statsmodels IRLS): {sum(c['pass'] for c in audit['checks'])}/{len(audit['checks'])} "
      f"checks pass; placebo {audit['placebo_model_swap_independent']}")

ra = A["random_arm"]
print(f"\nrandom-direction control on {ra['n_core_items']} items (change in EN refusal vs lambda 0):")
print(pd.DataFrame([{"rank": k, "model": m, "lambda": v["per_model"][m]["lambda"],
                     "dEN_heretic": v["per_model"][m]["dEN_heretic"], "dEN_rand": v["per_model"][m]["dEN_rand"],
                     "full dEN_heretic": REF["random_arm_dEN"][k][m]["dEN_heretic"],
                     "full dEN_rand": REF["random_arm_dEN"][k][m]["dEN_rand"]}
                    for k, v in ra["by_rank"].items() for m in MODELS]).to_string(index=False))

# ---------- figure
fig, axs = plt.subplots(1, 3, figsize=(15, 4.3))
ax = axs[0]
for m in MODELS:
    for s in A["steps_R"][m]:
        e = wil(s["k_sl"], s["n_sl"])
        mk = "s" if s["lambda"] == 0 else "o"
        ax.errorbar(s["p_en"], s["p_sl"], yerr=[[e[0]], [e[1]]], fmt=mk, color=COL[m], ms=4, lw=0.8,
                    mfc="white" if s["lambda"] == 0 else COL[m])
    pm = R["per_model"][m]
    xs = np.linspace(0.02, 0.98, 100)
    ax.plot(xs, expit(pm["a"]["point"] + pm["b"]["point"] * _lg(xs)), color=COL[m], lw=1.4,
            label=f"{NAME[m]}: a={pm['a']['point']:.2f}, b={pm['b']['point']:.2f}")
ax.plot([0, 1], [0, 1], ":", color="grey", lw=0.8)
ax.axvline(0.5, color="grey", lw=0.6, ls="--")
ax.set_xlabel("EN-BT refusal rate at step (J1, R)")
ax.set_ylabel("SL-MT refusal rate at step")
ax.set_title(f"Dose ladder, {A['n_items_body']}-item slice")
ax.legend(fontsize=7, loc="upper left")

ax = axs[1]
keys = ["G3", "G3_orig", "G3_edit"]
xx = np.arange(len(keys))
for off, (lab, src, c) in enumerate([(f"slice n={A['n_items_body']}", R["RAW"], "#555555"),
                                     (f"full n={REF['n_items_body']}", REF["headline_R_RAW"], "#2ca02c")]):
    pts = np.array([src[k]["point"] for k in keys])
    lo = np.array([src[k]["ci95"][0] for k in keys])
    hi = np.array([src[k]["ci95"][1] for k in keys])
    ax.errorbar(xx + (off - 0.5) * 0.25, pts, yerr=[pts - lo, hi - pts], fmt="o", color=c, capsize=3, label=lab)
ax.axhline(0, color="grey", lw=0.8)
ax.set_xticks(xx, ["G3 (total)", "G3_orig (λ=0)", "G3_edit (edit-induced)"])
ax.set_ylabel("GaMS − Gemma, log-odds (95% CI)")
ax.set_title("Lag decomposition: G3 = G3_orig + G3_edit")
ax.legend(fontsize=8)

ax = axs[2]
for m in MODELS:
    ks = sorted(ra["by_rank"], key=int)
    lam = [ra["by_rank"][k]["per_model"][m]["lambda"] for k in ks]
    ax.plot(lam, [ra["by_rank"][k]["per_model"][m]["dEN_heretic"] for k in ks], "o-", color=COL[m],
            label=f"{NAME[m]} Heretic")
    ax.plot(lam, [ra["by_rank"][k]["per_model"][m]["dEN_rand"] for k in ks], "x--", color=COL[m],
            label=f"{NAME[m]} random")
ax.axhline(0, color="grey", lw=0.8)
ax.set_xlabel("λ")
ax.set_ylabel("Δ EN refusal vs λ = 0")
ax.set_title("Specificity: Heretic vs norm-matched random edit")
ax.legend(fontsize=7)
plt.tight_layout()
plt.show()